In [2]:
# --- Standard library ---
from pathlib import Path
import os, json, re, ast, gc, warnings
from datetime import timezone
from typing import List, Dict, Tuple

# --- Data / utils ---
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- NLP / embeddings ---
from sentence_transformers import SentenceTransformer  # SBERT embeddings

# --- ANN (fast k-NN) ---
import hnswlib  # cosine ANN index

# --- Sentiment (baseline) ---
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer  # VADER
# NOTE: run once in a separate cell: nltk.download("vader_lexicon")

# --- (Optional) small helpers ---
from sklearn.preprocessing import MinMaxScaler  # if you prefer sklearn for scaling


In [3]:
DATA = Path("../data")  # change if your files live elsewhere
MOVIES_PATH  = DATA / "processed/movie_df.csv"        # merged TMDb metadata
REVIEWS_PATH = DATA / "raw/reviews.csv"    # TMDb reviews you downloaded (CSV)

# --- Load movies ---
movies = pd.read_csv(MOVIES_PATH, low_memory=False)

# Normalize TMDb ID
movies["id"] = pd.to_numeric(movies["id"], errors="coerce")
movies = movies.dropna(subset=["id"]).copy()
movies["id"] = movies["id"].astype(int)

# Ensure expected columns exist (create empty ones if missing)
for col in ["title", "overview", "genres", "popularity", "poster_path", "release_date", "year"]:
    if col not in movies.columns:
        movies[col] = pd.NA

# Derive year if not present
if movies["year"].isna().all():
    movies["year"] = pd.to_datetime(movies["release_date"], errors="coerce").dt.year

# Drop rows with completely empty overview (optional but recommended for embeddings)
movies = movies[movies["overview"].fillna("").str.strip().ne("")].reset_index(drop=True)

# De-duplicate by TMDb id just in case
movies = movies.drop_duplicates(subset=["id"]).reset_index(drop=True)

print(f"Movies loaded: {len(movies):,}")
display(movies.head(3))

# --- Load reviews (CSV) ---
reviews = pd.read_csv(REVIEWS_PATH, low_memory=False)

# Standardize ID column → tmdb_id
if "tmdb_id" not in reviews.columns:
    if "id" in reviews.columns:
        reviews = reviews.rename(columns={"id": "tmdb_id"})
    else:
        raise ValueError("Reviews CSV must contain 'tmdb_id' (or 'id') column.")

# Types & basic cleaning
reviews["tmdb_id"] = pd.to_numeric(reviews["tmdb_id"], errors="coerce")
reviews = reviews.dropna(subset=["tmdb_id"]).copy()
reviews["tmdb_id"] = reviews["tmdb_id"].astype(int)

# Text column name should be 'content'
if "content" not in reviews.columns:
    raise ValueError("Reviews CSV must contain a 'content' text column.")

reviews["content"] = reviews["content"].fillna("").astype(str).str.strip()
reviews = reviews[reviews["content"].str.len() > 0].copy()

# Normalize language column (if present) and optionally keep English only
if "iso_639_1" in reviews.columns and "lang" not in reviews.columns:
    reviews = reviews.rename(columns={"iso_639_1": "lang"})

# Parse timestamps if present
if "created_at" in reviews.columns:
    reviews["created_at"] = pd.to_datetime(reviews["created_at"], errors="coerce", utc=True)
else:
    reviews["created_at"] = pd.NaT

print(f"Reviews loaded: {len(reviews):,} (unique movies with reviews: {reviews['tmdb_id'].nunique():,})")
display(reviews.head(3))

# --- Quick join sanity check ---
covered = movies["id"].isin(reviews["tmdb_id"]).sum()
print(f"Movies with ≥1 review: {covered:,} / {len(movies):,}")


Movies loaded: 44,471


,adult,genres,id,overview,popularity,poster_path,production_countries,release_date,spoken_languages,title,vote_average,vote_count,keywords,cast,year
0,False,"['Comedy', 'Crime']",100,A card sharp and his unwillingly-enlisted frie...,4.607860,/qV7QaSf7f7yC2lc985zfyOJIAIN.jpg,['United Kingdom'],1998-03-05,['English'],"Lock, Stock and Two Smoking Barrels",7.5,1671.0,"['ambush', 'alcohol', 'shotgun', 'tea', 'joint...","['Jason Flemyng', 'Dexter Fletcher', 'Nick Mor...",1998.0
1,False,"['Comedy', 'Drama']",10000,A group of tenants living in an old house are ...,0.281609,/mdfFmAd1dXPa02GvZTnGDPWLut6.jpg,"['Colombia', 'France', 'Italy']",1993-12-25,['Español'],La estrategia del caracol,7.2,9.0,"['roommate', 'pastor', 'squatter', 'anarchist'...",[],1993.0
2,False,"['Comedy', 'Science Fiction']",10001,Albert Einstein is the son of a Tasmanian appl...,2.562888,/5urimrfyzWmpcZErQ0jH4uLHdd.jpg,['Australia'],1988-12-15,['English'],Young Einstein,4.5,46.0,"['atomic bomb', 'nobel prize', 'rock', 'albert...","['Yahoo Serious', 'Odile Le Clezio', 'Peewee W...",1988.0


Reviews loaded: 18,404 (unique movies with reviews: 10,473)


,tmdb_id,review_id,author,username,rating,lang,created_at,url,content
0,211672,55a58e46c3a3682bb2000065,Andres Gomez,tanty,4.0,NaN,2015-07-14 22:33:42.469000+00:00,https://www.themoviedb.org/review/55a58e46c3a3...,The minions are a nice idea and the animation ...
1,211672,55e108c89251416c0b0006dd,movizonline.com,movizonline,NaN,NaN,2015-08-29 01:20:08.744000+00:00,https://www.themoviedb.org/review/55e108c89251...,a nice idea and the animation.the new thing in...
2,211672,59c38385c3a368141e00e1f2,Tom Morvolo Riddle,Tom Morvolo Riddle,NaN,NaN,2017-09-21 09:16:53.123000+00:00,https://www.themoviedb.org/review/59c38385c3a3...,I enjoyed it a lot with my nephews. It's a fun...


Movies with ≥1 review: 10,473 / 44,471


Review sentiment (per review) → VADER (markdown cell)

We’ll score each review using VADER’s compound score in [-1,1], then later aggregate with recency weighting.

In [4]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download("vader_lexicon", quiet=True)

sia = SentimentIntensityAnalyzer()
reviews["sent_vader"] = reviews["content"].map(lambda t: sia.polarity_scores(t)["compound"]).astype(float)


Aggregate to movie-level sentiment & keep top review snippets (markdown cell)

Recency weight: exponential decay (half-life ≈ 18 months)

Smoothing: shrink towards neutral (0.0) when few reviews

Top snippets: concatenate the most recent N reviews (bounded by chars)

In [6]:
import numpy as np
import pandas as pd
from datetime import timezone

now = pd.Timestamp.now(tz=timezone.utc)

def recency_weight(ts):
    if pd.isna(ts): return 1.0
    days = (now - ts).days
    return 0.5 ** (days / 540.0)  # ~18 months half-life

reviews["w"] = reviews["created_at"].map(recency_weight)

def aggregate_reviews_text(g, max_reviews=5, max_chars=8000):
    g = g.sort_values("created_at", ascending=False)
    texts, total = [], 0
    for t in g["content"]:
        if not isinstance(t, str): 
            continue
        t = t.strip()
        if not t:
            continue
        if total + len(t) > max_chars:
            break
        texts.append(t)
        total += len(t)
        if len(texts) >= max_reviews:
            break
    return "\n\n".join(texts)

# Bayesian smoothing towards 0.0 (neutral) for low review_count
def smoothed_mean(values, weights, prior_mean=0.0, alpha=5.0):
    v = np.asarray(values, float)
    w = np.asarray(weights, float)
    if np.sum(w) == 0:
        return prior_mean
    raw = np.average(v, weights=w)
    n = len(v)
    return (n * raw + alpha * prior_mean) / (n + alpha)

agg = (
    reviews.groupby("tmdb_id").apply(
        lambda g: pd.Series({
            "review_count": int(len(g)),
            "review_mean_sent": float(smoothed_mean(g["sent_vader"], g["w"], prior_mean=0.0, alpha=5.0)),
            "review_std_sent": float(g["sent_vader"].std(ddof=0)) if len(g) > 1 else 0.0,
            "reviews_text": aggregate_reviews_text(g, max_reviews=5, max_chars=8000),
        })
    )
    .reset_index().rename(columns={"tmdb_id": "id"})
)
agg.head(3)


/tmp/ipykernel_577831/550385772.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  reviews.groupby("tmdb_id").apply(


,id,review_count,review_mean_sent,review_std_sent,reviews_text
0,3,1,0.156950,0.000000,"Released in 1986, Aki Kaurismaki's <i>Varjoja ..."
1,11,6,0.529890,0.019018,Thinking back to the films that define my gene...
2,12,4,0.376866,0.257700,"""Nemo"" is your typically adventurous and curio..."


Merge aggregates → build text_full (markdown cell)

We’ll merge agg into movies, and build the text we’ll embed. If genres is stored as a Python-literal string (e.g., "[{'id':12,'name':'Adventure'}]"), we’ll extract names.

In [7]:
import ast

movies_aug = movies.merge(agg, on="id", how="left")

movies_aug["review_count"] = movies_aug["review_count"].fillna(0).astype(int)
movies_aug["review_mean_sent"] = movies_aug["review_mean_sent"].fillna(0.0).astype(float)
movies_aug["review_std_sent"]  = movies_aug["review_std_sent"].fillna(0.0).astype(float)
movies_aug["reviews_text"]     = movies_aug["reviews_text"].fillna("")

def genres_to_text(x):
    if isinstance(x, list):
        # list of dicts or strings
        if x and isinstance(x[0], dict):
            return " ".join(sorted(d.get("name","") for d in x if isinstance(d, dict))).strip()
        return " ".join(str(t) for t in x)
    if isinstance(x, str):
        s = x.strip()
        # try to parse python-literal form
        if s.startswith("[") and ("]" in s):
            try:
                lst = ast.literal_eval(s)
                if isinstance(lst, list):
                    if lst and isinstance(lst[0], dict):
                        return " ".join(sorted(d.get("name","") for d in lst if isinstance(d, dict))).strip()
                    return " ".join(str(t) for t in lst)
            except Exception:
                pass
        return s
    return ""

movies_aug["genres_text"] = movies_aug["genres"].apply(genres_to_text)

def make_text(row):
    parts = [
        str(row.get("title","")),
        str(row.get("overview","")),
        row.get("genres_text",""),
    ]
    if row.get("reviews_text"):
        parts.append(row["reviews_text"])
    return " . ".join(p for p in parts if isinstance(p, str) and p.strip())

movies_aug["text_full"] = movies_aug.apply(make_text, axis=1)
# drop rows with empty text
movies_aug = movies_aug[movies_aug["text_full"].str.strip().ne("")].reset_index(drop=True)

len(movies_aug), movies_aug[["id","title","year","review_count"]].head(3)


(44471,
       id                                title    year  review_count
 0    100  Lock, Stock and Two Smoking Barrels  1998.0             4
 1  10000            La estrategia del caracol  1993.0             0
 2  10001                       Young Einstein  1988.0             0)

Embeddings with Sentence-Transformers (markdown cell)

Encode text_full using all-MiniLM-L6-v2 with normalized embeddings (so cosine similarity = inner product).

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
st_model = SentenceTransformer(EMB_MODEL)

emb = st_model.encode(
    movies_aug["text_full"].tolist(),
    batch_size=128,
    normalize_embeddings=True,
    show_progress_bar=True
).astype("float32")

emb.shape


Batches: 100%|██████████| 348/348 [00:20<00:00, 17.40it/s]


(44471, 384)

Build ANN index (HNSW, cosine) (markdown cell)

We’ll use hnswlib for fast k-NN over embeddings and save it to disk.

In [10]:
import hnswlib
from pathlib import Path

ART = Path("../artifacts"); ART.mkdir(exist_ok=True)

N, d = emb.shape
index = hnswlib.Index(space="cosine", dim=d)
index.init_index(max_elements=N, ef_construction=200, M=16)
index.add_items(emb, np.arange(N))
index.set_ef(200)  # higher → better recall, slower query
index.save_index(str(ART / "hnsw_cosine.idx"))


Save artifacts (markdown cell)

Persist everything your Streamlit app will load.

In [11]:
import json
import numpy as np
import pandas as pd

# embeddings
np.save(ART / "embeddings.npy", emb)

# movie-level sentiment in [0,1]
sent01 = ((movies_aug["review_mean_sent"].values + 1.0) / 2.0).astype("float32")
np.save(ART / "review_sentiment_01.npy", sent01)

# data table
movies_aug.to_parquet(ART / "movies_aug.parquet", index=False)

# id maps
id_map = {
    "tmdb_ids": movies_aug["id"].astype(int).tolist(),
    "row_by_tmdb": {int(mid): int(i) for i, mid in enumerate(movies_aug["id"].astype(int))}
}
with open(ART / "id_map.json", "w") as f:
    json.dump(id_map, f)

# config
config = {
    "embedding_model": EMB_MODEL,
    "n_movies": int(N),
    "emb_dim": int(d),
    "created_at": pd.Timestamp.utcnow().isoformat()
}
with open(ART / "config.json", "w") as f:
    json.dump(config, f, indent=2)

# quick confirmation
print("✅ wrote:", [p.name for p in ART.iterdir()])


✅ wrote: ['config.json', 'review_sentiment_01.npy', 'movies_aug.parquet', 'id_map.json', 'embeddings.npy', 'hnsw_cosine.idx']


Smoke test for neighbours (markdown cell)

Quick local check: seed by a title or by free text, retrieve ANN neighbours, and fuse with sentiment.

In [12]:
from rapidfuzz import process, fuzz

def knn_query(index, qv, k=20):
    labels, distances = index.knn_query(qv.astype("float32"), k=k)
    idxs = labels[0]
    sims = 1.0 - distances[0]  # cosine similarity
    return idxs, sims

def normalize01(x):
    x = x.astype("float32")
    mn, mx = float(np.nanmin(x)), float(np.nanmax(x))
    return np.zeros_like(x) if mx <= mn else (x - mn) / (mx - mn + 1e-9)

def fuse(sim, ids, w_sim=0.7, w_sent=0.25, w_pop=0.05):
    sim01 = normalize01(sim)
    s = sent01[ids]
    pop = movies_aug.iloc[ids]["popularity"].values if "popularity" in movies_aug.columns else None
    parts = [w_sim*sim01, w_sent*s]
    if pop is not None: parts.append(w_pop*normalize01(pop))
    return np.sum(np.vstack(parts), axis=0)

# by title (fuzzy match)
seed = "John Wick"
match = process.extractOne(seed, movies_aug["title"].tolist(), scorer=fuzz.WRatio)
if match and match[1] >= 60:
    seed_title = match[0]
    i = int(movies_aug.index[movies_aug["title"] == seed_title][0])
    idxs, sims = knn_query(index, emb[i:i+1], k=50)
    # drop self if first
    if idxs[0] == i:
        idxs, sims = idxs[1:], sims[1:]
    scores = fuse(sims, idxs)
    top = np.argsort(-scores)[:10]
    print("Seed:", seed_title)
    display(movies_aug.iloc[idxs[top]][["title","year","review_count"]])
else:
    print("Seed title not found with enough confidence.")

# by free text
q = "neon-noir revenge, balletic gun-fu, minimal dialogue"
qv = st_model.encode([q], normalize_embeddings=True).astype("float32")
idxs, sims = knn_query(index, qv, k=10)
display(movies_aug.iloc[idxs][["title","year","review_count"]])


Seed: John Wick


,title,year,review_count
23148,John Wick: Chapter 2,2017.0,11
23865,Criminal Activities,2015.0,1
43074,Phenomenon,1996.0,1
9041,Speed,1994.0,4
42281,Conspiracy Theory,1997.0,2
23302,Extraction,2015.0,3
24465,Keanu,2016.0,1
24286,The Circle,2017.0,3
21947,Blood Father,2016.0,3
39272,Territories,2010.0,1


,title,year,review_count
1981,House of the Dead,2003.0,1
4134,Film Is a Girl & a Gun,2009.0,0
43709,Vengeance,1968.0,1
23185,Cute and Dangerous,2015.0,0
16660,Kung Fury,2015.0,1
31013,Zero 3,2017.0,0
17522,Bullet Ballet,1998.0,0
21455,The Hunter's Prayer,2017.0,1
36215,Lipstick,1976.0,0
35596,Once,2007.0,2
